# Lesson 2.1 — Designing Tool Interfaces with Clear Descriptions & Boundaries

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain why tool descriptions are the primary mechanism LLMs use for tool selection
- Write descriptions with input formats, example queries, edge cases, and boundary explanations
- Eliminate overlap by renaming tools and splitting generic tools into purpose-specific ones
- Audit system prompts for keyword-sensitive wording that overrides good descriptions

**You will produce:** A differentiated tool pair (get_customer vs lookup_order) plus a split of a generic tool.

## Phase 2 Overview

Phase 2 covers **Domain 2 - Tool Design & MCP Integration (18%)**. You will design tool interfaces the model can select reliably, return structured MCP errors, scope tools and configure tool_choice, integrate MCP servers via .mcp.json, and apply the built-in tools.

By the end of this phase, you will have built:

- Differentiated, boundary-rich tool descriptions (Lesson 2.1)
- Structured MCP error responses with errorCategory/isRetryable (Lesson 2.2)
- Scoped tool distribution and tool_choice configuration (Lesson 2.3)
- A project `.mcp.json` with env expansion and resources (Lesson 2.4)
- A built-in tool selection playbook: Read/Write/Edit/Bash/Grep/Glob (Lesson 2.5)

### Prerequisites

- Completion of Phase 1 (agentic loops, subagents, hooks)
- Python 3 kernel with the `anthropic` SDK; optional `ANTHROPIC_API_KEY`
- Familiarity with JSON Schema and the Model Context Protocol (MCP)

## Concepts

**Tool descriptions are the primary mechanism LLMs use for tool selection.** Minimal descriptions ("Retrieves customer information" / "Retrieves order details") lead to misrouting - Sample Question 2 shows the agent calling `get_customer` for order queries. The highest-leverage first fix is to **expand each description** with:

- the **input formats** it accepts,
- **example queries** that should route to it,
- **edge cases**, and
- **boundaries** explaining when to use it vs a similar tool.

Other techniques:
- **Rename + redescribe** to remove functional overlap (e.g. `analyze_content` -> `extract_web_results` with a web-specific description).
- **Split a generic tool** into purpose-specific tools with defined I/O contracts (e.g. `analyze_document` -> `extract_data_points`, `summarize_content`, `verify_claim_against_source`).
- **Audit system prompts** for keyword-sensitive instructions that create unintended tool associations and override well-written descriptions.

> **Reference:** [Anthropic - Tool use (defining tools)](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — From minimal to boundary-rich descriptions

We rewrite two confusable tools so each states inputs, examples, edge cases, and boundaries.

In [ ]:
get_customer = {
    "name": "get_customer",
    "description": (
        "Look up a CUSTOMER account by customer identifier (email, phone, or account ID). "
        "Use for identity verification and profile data. "
        "Example queries: 'find my account', 'verify me by email'. "
        "Do NOT use for order status - use lookup_order. "
        "Edge case: multiple matches -> return all and ask for another identifier."
    ),
    "input_schema": {"type": "object",
        "properties": {"identifier": {"type": "string"}},
        "required": ["identifier"]},
}
lookup_order = {
    "name": "lookup_order",
    "description": (
        "Retrieve ORDER details by order number (format: #NNNNN). "
        "Use for status, items, totals, and shipment info. "
        "Example queries: 'check my order #12345', 'where is order 12345'. "
        "Requires a verified customer_id from get_customer first."
    ),
    "input_schema": {"type": "object",
        "properties": {"order_number": {"type": "string"},
                         "customer_id": {"type": "string"}},
        "required": ["order_number", "customer_id"]},
}
print(get_customer['description'][:90], '...')
print(lookup_order['description'][:90], '...')

## Code-Along: Step 2 — Split a generic tool into purpose-specific tools

`analyze_document` is too broad. We split it into three tools with defined contracts.

In [ ]:
split_tools = {
    "extract_data_points": "Return structured fields from a document (JSON schema).",
    "summarize_content": "Return a concise prose summary of a document.",
    "verify_claim_against_source": "Check a specific claim against a cited source.",
}
for name, desc in split_tools.items():
    print(f"{name}: {desc}")

## Code-Along: Step 3 — Audit a system prompt for keyword bias

A stray keyword can override good descriptions. We flag risky phrasing.

In [ ]:
system_prompt = "You are a support agent. For any customer message, first get customer info."
risky = [kw for kw in ["customer info", "get customer"] if kw in system_prompt.lower()]
print("Keyword associations that may force get_customer:", risky)
# Fix: describe WHEN to verify identity vs look up an order, not a blanket keyword rule.

## Exercise

You have two near-identical tools `analyze_content` and `analyze_document`. Rename and redescribe them to remove overlap (one web-specific, one document-specific), each with input format, an example query, an edge case, and an explicit boundary line. Then write a tiny checker that fails any description shorter than 120 characters.

In [ ]:
# TODO: define extract_web_results and extract_document_data with rich descriptions
# TODO: assert len(desc) >= 120 for each tool description

## Key Takeaways

- Tool descriptions are the primary tool-selection mechanism - make them rich, not minimal.
- Include input formats, example queries, edge cases, and explicit boundaries vs similar tools.
- Rename/redescribe to remove overlap; split generic tools into purpose-specific ones.
- Audit system prompts for keyword associations that can override good descriptions.

---

**Next:** [Lesson 2.2 — Structured Error Responses for MCP Tools](./lesson-2.2-structured-mcp-errors.ipynb)